In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, classification_report

# Suport functions

In [24]:
def confusion(true, pred):
    """
    Function for pretty printing confusion matrices
    """
    true.name = 'target'
    pred.name = 'predicted'
    cm = pd.crosstab(true.reset_index(drop=True), pred.reset_index(drop=True))
    cm = cm[cm.index]
    return cm

# Data loading

In [25]:
ILDS = pd.read_csv("log_ILDS_train_X.csv", delimiter=',')
ILDS.columns = ['Age','TP','ALB','AR','DBratio','logTB','logDB','logAlkphos','logSgpt','logSgot','Female', 'Target']

ILDS.head()

,Age,TP,ALB,AR,DBratio,logTB,logDB,logAlkphos,logSgpt,logSgot,Female,Target
0,-0.369595,0.936798,1.533612,1.522563,-0.907002,0.229163,0.473897,-0.571139,0.120587,0.337926,0,0
1,-1.346302,-0.215915,-0.010528,0.222596,-0.009897,-0.428872,-0.381170,-0.213474,0.088368,0.642955,0,0
2,-0.186462,-0.215915,0.118150,0.427854,0.112435,-0.794539,-0.696750,-0.940710,-0.123185,0.657821,1,0
3,0.423980,0.456501,0.246829,-0.119501,1.947424,-0.952576,-1.236238,-0.407418,-1.798188,-1.627323,1,1
4,-0.003330,-0.792272,-0.782599,-0.461597,-0.866225,-0.159894,0.158317,-0.748123,0.326694,-0.128145,0,1


In [26]:
X = ILDS.loc[:, ILDS.columns != 'Target']
y = ILDS['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [27]:
results_df = pd.DataFrame(index=[], columns= ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'])

# Random forest

In [28]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)


In [29]:
cross_val_results = pd.DataFrame(cross_validate(rf , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['RF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.743756,0.743168,0.74704,0.744042


# Gradient Boosting

In [30]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

In [31]:
cross_val_results = pd.DataFrame(cross_validate(gb , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['GB',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.743756,0.743168,0.74704,0.744042
GB,0.741405,0.741022,0.743577,0.741783


# Voting classifier

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

log_clf = LogisticRegression()
svc_clf = SVC(probability=True)
voting = VotingClassifier(estimators=[
    ('lr', log_clf), ('svc', svc_clf), ('rf', rf)
], voting='soft')

voting.fit(X_train, y_train)
voting_pred = voting.predict(X_test)


In [33]:
cross_val_results = pd.DataFrame(cross_validate(log_clf , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['CLF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.743756,0.743168,0.74704,0.744042
GB,0.741405,0.741022,0.743577,0.741783
CLF,0.666676,0.664313,0.67291,0.667979


In [35]:
ILDS_test = pd.read_csv("log_ILDS_test_X.csv", delimiter=',', header=None)

ILDS_test.columns = ['Age','TP','ALB','AR','DBratio','logTB','logDB','logAlkphos','logSgpt','logSgot','Female']

X_test = ILDS_test.loc[:,:'Female']

ILDS_test['Label'] = rf.predict(X_test)

ILDS_test.head()

ILDS_test.index = ILDS_test.index + 1
ILDS_test.index.name = 'ID'

ILDS_test['Label'].to_csv('random_forest.csv', index=True)